# TF Causal Model — Per-(perturbation, TF) Activity Z-scores

For each (non-TF perturbation, RegulonDB TF) pair, score whether the TF's regulon behaves as the model predicts:
1. Train a causal (RegulonDB-masked) linear NB model on TF-knockdown cells
2. For each held-out perturbation, aggregate signed NB residuals over each TF's regulon → signed Z ~ N(0,1) under "regulon behaves as predicted"
3. Per-TF robust recalibration (median / MAD) of Z across perturbations

In [ ]:
import sys

sys.path.insert(0, "/workspace/src")

import numpy as np
import pandas as pd
import scanpy as sc
import jax
import jax.numpy as jnp
import flax.linen as nn
import json
import optax
from flax.linen.initializers import glorot_normal, zeros
from flax.training import train_state
from numpyro.distributions import NegativeBinomial2
from scipy import sparsetf_cau
from scipy.stats import median_abs_deviation
from scipy import stats
from tqdm import tqdm
from essential.data import load_regulondb_full

In [ ]:
ADATA_PATH = "/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad"
PERT_COL = "target"
CTRL_KEY = "nontargeting"
EXPERIMENT_SUBSET = "lce75"
MIN_LIB = 1e3ƒ

N_EPOCHS = 100
BATCH_SIZE = 512
LR = 1e-3

OUT_PATH = "/workspace/experiments/06152026_retreat_content/causal_zscores.parquet"

## 1. Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata.obs["_lib"] = np.asarray(adata.layers["reads"].sum(1)).ravel()
adata = adata[
    (adata.obs["_lib"] > MIN_LIB)
    & (adata.obs["experiment"] == EXPERIMENT_SUBSET)
    & adata.obs[PERT_COL].notna()
].copy()
adata.var_names = adata.var_names.str.lower()
adata.obs[PERT_COL] = adata.obs[PERT_COL].str.lower()

raw = adata.layers["reads"]
raw = raw.toarray() if sparse.issparse(raw) else np.asarray(raw, np.float32)
lib_col = raw.sum(1, keepdims=True)
adata.layers["counts"] = raw.astype(np.float32)
adata.layers["lcp10k"] = np.log1p(raw / (lib_col + 1e-6) * 1e4).astype(np.float32)
adata.X = adata.layers["counts"]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
ref_db = load_regulondb_full()
ref_db = ref_db[ref_db["ri_type"].str.startswith("TF")].copy()

## Filter on confidence level
# ref_db = ref_db.loc[lambda x: x["confidenceLevel"] != "W"].copy()

tf_info = (
    ref_db.assign(TF=ref_db["regulator_gene"].str.lower())
    .groupby("TF")
    .agg(
        n_edges=("target_gene", "size"),
        frac_uncertain=("confidenceLevel", lambda s: (s == "W").mean()),
        frac_nan=("confidenceLevel", lambda s: s.isna().mean()),
    )
    .reset_index()
)
## Filter on number of edges
valid_tfs = tf_info.query("n_edges >= 2")["TF"].to_list()
ref_db = ref_db[ref_db["regulator_gene"].str.lower().isin(valid_tfs)].copy()

all_tfs = set(ref_db["regulator_gene"].str.lower().unique())

perts = adata.obs[PERT_COL]
obs_targets = set(perts.unique()) - {CTRL_KEY}
tf_perts = obs_targets & all_tfs
non_tf_perts = obs_targets - all_tfs

adata_train = adata[perts.isin(tf_perts) | (perts == CTRL_KEY)].copy()
adata_val = adata[perts.isin(non_tf_perts)].copy()

print(
    f"Train: {adata_train.n_obs:,} cells  ({len(tf_perts)} TF perturbations + controls)"
)
print(f"Val:   {adata_val.n_obs:,} cells  ({len(non_tf_perts)} non-TF perturbations)")

In [ ]:
ref_db.loc[lambda x: x["regulator_gene"] == "rpos"]

In [ ]:
var_names = list(adata.var_names)
gene_idx = {g: i for i, g in enumerate(var_names)}
tf_genes = sorted(g for g in all_tfs if g in gene_idx)
tf_idx_map = {g: i for i, g in enumerate(tf_genes)}
tf_cols = np.array([gene_idx[g] for g in tf_genes])
n_genes, n_tfs = adata.n_vars, len(tf_genes)

# Amask_tf[i, k] = 1  <=>  TF k regulates gene i  (RegulonDB)
Amask_tf = np.zeros((n_genes, n_tfs), dtype=np.float32)
for _, row in ref_db.iterrows():
    t, r = row["target_gene"].lower(), row["regulator_gene"].lower()
    if t in gene_idx and r in tf_idx_map:
        Amask_tf[gene_idx[t], tf_idx_map[r]] = 1.0

# restrict to genes with >=1 known TF regulator
has_reg = Amask_tf.sum(1) > 0  # (n_genes,) bool
reg_idx = np.where(has_reg)[0]  # indices into full gene list
reg_names = [var_names[i] for i in reg_idx]
n_reg = len(reg_idx)

print(f"{n_tfs} TF features  |  {int(Amask_tf.sum()):,} RegulonDB edges")
print(f"{n_reg} genes with >=1 TF regulator (analysis set)")

In [ ]:
def get_arrays(a):
    lcp = np.asarray(a.layers["lcp10k"], dtype=np.float32)
    raw = np.asarray(a.layers["counts"], dtype=np.float32)
    return lcp, raw


lcp_train, raw_train = get_arrays(adata_train)
lcp_val, raw_val = get_arrays(adata_val)

ctrl_mask = np.asarray(adata_train.obs[PERT_COL] == CTRL_KEY)
tf_mu = lcp_train[ctrl_mask][:, tf_cols].mean(0)
tf_sigma = lcp_train[ctrl_mask][:, tf_cols].std(0)
tf_sigma = np.where(tf_sigma > 1e-3, tf_sigma, 1.0)

Xtrain = (lcp_train[:, tf_cols] - tf_mu) / tf_sigma
Xval = (lcp_val[:, tf_cols] - tf_mu) / tf_sigma
lib_train = raw_train.sum(1)
lib_val = raw_val.sum(1)

ctrl_lcp_mean = lcp_train[ctrl_mask].mean(0)  # (n_genes,)

## 2. Model

In [ ]:
class TFLinearNB(nn.Module):
    n_genes: int
    n_tfs: int
    x_mean: jnp.ndarray  # (n_genes,) frozen control log-CP10K mean
    Amask_tf: jnp.ndarray  # (n_genes, n_tfs)

    @nn.compact
    def __call__(self, x_tf, y_raw, lib):
        W = self.param("W", glorot_normal(), (self.n_genes, self.n_tfs))
        b = self.param("b", zeros, (self.n_genes,))
        overdispersion_ = self.param("overdispersion_", zeros, (self.n_genes,))

        mask = jax.lax.stop_gradient(self.Amask_tf)
        x_mean = jax.lax.stop_gradient(jnp.array(self.x_mean))
        lcp_pred = x_mean + x_tf @ (W * mask).T + b
        lib_scale = lib[:, None] / 1e4
        mean = jnp.maximum(jnp.expm1(lcp_pred) * lib_scale, 1e-8)
        conc = jnp.exp(overdispersion_)
        nll = -NegativeBinomial2(mean=mean, concentration=conc).log_prob(y_raw).mean()
        return {"loss": nll, "nll": nll}

## 3. Training

In [ ]:
def make_step(model):
    @jax.jit
    def step(state, x, y, lib):
        def loss_fn(params):
            out = model.apply({"params": params}, x, y, lib)
            return out["loss"], out

        (_, out), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        return state.apply_gradients(grads=grads), out

    return step


def train(model, X, Y, lib, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR):
    key = jax.random.PRNGKey(0)
    params = model.init(key, jnp.array(X[:4]), jnp.array(Y[:4]), jnp.array(lib[:4]))[
        "params"
    ]
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=optax.adam(lr)
    )
    step = make_step(model)
    rng, history = np.random.default_rng(0), []
    n = len(X)
    for _ in tqdm(range(n_epochs)):
        idx = rng.permutation(n)
        losses = []
        for s in range(0, n - batch_size + 1, batch_size):
            b = idx[s : s + batch_size]
            state, out = step(
                state, jnp.array(X[b]), jnp.array(Y[b]), jnp.array(lib[b])
            )
            losses.append(float(out["nll"]))
        history.append(np.mean(losses))
    return state, history

In [ ]:
x_mean_frozen = jnp.array(ctrl_lcp_mean)

model_cau = TFLinearNB(
    n_genes=n_genes,
    n_tfs=n_tfs,
    x_mean=x_mean_frozen,
    Amask_tf=jnp.array(Amask_tf),
)

print("Training causal (RegulonDB) model...")
state_cau, hist_cau = train(model_cau, Xtrain, raw_train, lib_train)
print(f"Final train NLL: {hist_cau[-1]:.4f}")

In [ ]:
plt.plot(hist_cau)

In [ ]:
pert_labels = np.array(adata_val.obs[PERT_COL])
unique_perts, pert_idx = np.unique(pert_labels, return_inverse=True)
n_perts = len(unique_perts)
n_val = len(Xval)

_W = jnp.array(np.array(state_cau.params["W"]) * Amask_tf)  # (n_genes, n_tfs)
_b = jnp.array(state_cau.params["b"])  # (n_genes,)
_xm = jnp.array(ctrl_lcp_mean)  # (n_genes,)
_conc = jnp.exp(state_cau.params["overdispersion_"])  # (n_genes,)

## 4. TF-centric analysis

In [ ]:
# For each (perturbation p, TF t):
#     Z[p,t] = ( Σ_{i∈p} Σ_{g∈reg(t)} sign(W_gt)·(y_ig − μ_ig) )
#              / sqrt( Σ_{i∈p} Σ_{g∈reg(t)} V_ig ),     V = μ + μ²/conc
# Numerator: regulatory-sign-corrected sum of raw residuals over t's targets and p's cells.
# Denominator: the NB standard deviation of that sum. Z ~ N(0,1) under "regulon t behaves
# as predicted". Z > 0 ⇒ TF acts MORE active than its expression implies (de-repression /
# activation); Z < 0 ⇒ less. NB enters only through V.

# --- accumulate signed residual sum and NB-variance sum per (perturbation, gene) ---
Rnum = np.zeros((n_perts, n_genes), dtype=np.float64)  # Σ_{i∈p} (y - μ)
Vsum = np.zeros((n_perts, n_genes), dtype=np.float64)  # Σ_{i∈p} V


@jax.jit
def batch_resid(x_b, y_b, l_b):
    lcp = _xm + x_b @ _W.T + _b
    mu = jnp.maximum(jnp.expm1(lcp) * l_b[:, None] / 1e4, 1e-8)
    V = mu + mu**2 / _conc
    return y_b - mu, V


for s in tqdm(range(0, n_val, BATCH_SIZE), desc="TF residuals"):
    sl = slice(s, s + BATCH_SIZE)
    num_b, V_b = batch_resid(
        jnp.array(Xval[sl]), jnp.array(raw_val[sl]), jnp.array(lib_val[sl])
    )
    np.add.at(Rnum, pert_idx[sl], np.array(num_b, dtype=np.float64))
    np.add.at(Vsum, pert_idx[sl], np.array(V_b, dtype=np.float64))

# --- aggregate genes -> TFs (two matmuls) ---
Dsign = np.sign(np.asarray(_W))  # (n_genes, n_tfs): ±1 on edges, 0 elsewhere
#           swap Dsign -> np.asarray(_W) and Amask_tf -> np.asarray(_W)**2 below for the
#           magnitude-weighted (efficient) variant if you trust the learned W.
n_targets = Amask_tf.sum(0)  # (n_tfs,) targets per TF

Num = Rnum @ Dsign  # (n_perts, n_tfs) signed residual sum
Den = Vsum @ Amask_tf  # (n_perts, n_tfs) NB variance of that sum

with np.errstate(divide="ignore", invalid="ignore"):
    Z_tf = Num / np.sqrt(Den)  # signed, ~N(0,1) under null
Z_tf[:, n_targets == 0] = np.nan  # TFs with no targets in the gene set

# --- empirical per-TF recalibration (same robust trick as your gene-level z) ---
med = np.nanmedian(Z_tf, axis=0)
mad = median_abs_deviation(Z_tf, axis=0, scale="normal", nan_policy="omit")
mad = np.where(mad > 1e-8, mad, np.nan)
Z_emp = (Z_tf - med[None, :]) / mad[None, :]

# --- tidy table: TF, perturbation, Zscore --------------------------------------
# Z_tf is (n_perts, n_tfs); ravel() => perturbation slow, TF fast
df_tf = pd.DataFrame(
    {
        "TF": np.tile(np.array(tf_genes), n_perts),
        "perturbation": np.repeat(unique_perts, n_tfs),
        "Zscore": Z_tf.ravel().astype(np.float32),  # model-based signed N(0,1)
        "Zscore_emp": Z_emp.ravel().astype(np.float32),  # per-TF robust-standardized
        "n_targets": np.tile(n_targets.astype(int), n_perts),
    }
).dropna(subset=["Zscore"])
df_tf["abs_Zscore"] = df_tf["Zscore"].abs()
df_tf["abs_Zscore_emp"] = df_tf["Zscore_emp"].abs()

df_tf = df_tf.sort_values("abs_Zscore_emp", ascending=False).reset_index(drop=True)
df_tf

In [ ]:
print(f"TF-level table: {len(df_tf):,} (perturbation × TF) pairs")
pd.set_option("display.max_rows", 100)
df_tf.head(100)

In [ ]:
df_tf.loc[lambda x: x["TF"] == "sgrr"].sort_values("Zscore_emp", ascending=False).head(
    20
)

In [ ]:
bins = np.linspace(-6, 6, 101)

ax = df_tf["Zscore_emp"].hist(bins=bins, density=True)
x = np.linspace(-6, 6, 400)
ax.plot(x, stats.norm.pdf(x), color="k", lw=1.5, label=r"$\mathcal{N}(0,1)$")
ax.legend()
ax.set_xlabel("residual TF score")
ax.set_ylabel("density")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

z = df_tf["Zscore_emp"].abs().dropna().values
z_sorted = np.sort(z)
sf_emp = 1 - np.arange(1, len(z_sorted) + 1) / len(z_sorted)

t = np.logspace(-2, np.log10(z_sorted.max()), 500)
sf_hn = 2 * (1 - norm.cdf(t))  # P(|N(0,1)| > t)

fig, ax = plt.subplots()
ax.plot(z_sorted, sf_emp, label="empirical")
ax.plot(t, sf_hn, color="k", lw=1.5, label=r"$|\mathcal{N}(0,1)|$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|Zscore_emp|")
ax.set_ylabel("P(|Z| > t)")
ax.grid(True, which="both", ls="-", lw=0.5, alpha=0.6)
ax.legend()
plt.show()

In [ ]:
(df_tf["Zscore_emp"].abs() >= 10).sum()

In [ ]:
THRESHOLD = 10
df_tf_outliers = df_tf[df_tf["Zscore_emp"].abs() >= THRESHOLD]

display("Counting # of outliers per TF:")
n_outliers_per_tf = (
    df_tf_outliers["TF"]
    .value_counts()
    .reset_index(name="n_outliers")
    .rename(columns={"index": "TF"})
)
display(n_outliers_per_tf)

n_outliers_per_perturbation = (
    df_tf_outliers["perturbation"]
    .value_counts()
    .reset_index(name="n_outliers")
    .rename(columns={"index": "perturbation"})
)
display("Counting # of outliers per perturbation:")
display(n_outliers_per_perturbation)

In [ ]:
n_outliers_per_tf.head(50)

In [ ]:
top = n_outliers_per_tf.sort_values("n_outliers", ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top["TF"], top["n_outliers"])
ax.set_xticklabels(top["TF"], rotation=45)
ax.set_ylabel("# outlier perts")
ax.set_xlabel("TF")

In [ ]:
tf_name = "torr"
df_tf_ = df_tf.loc[lambda x: x["TF"] == tf_name]
plt.hist(df_tf_["Zscore"], bins=100, density=True)

In [ ]:
n_outliers_per_tf.hist(bins=30)

In [ ]:
df = n_outliers_per_tf.merge(tf_info, on="TF", how="left")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(df["n_edges"], df["n_outliers"], s=12, alpha=0.6)
# label the outliers-of-the-outlier-count
top = df.nlargest(15, "n_outliers")
for _, r in top.iterrows():
    ax.annotate(r["TF"], (r["n_edges"], r["n_outliers"]), fontsize=8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("regulon size (# edges)")
ax.set_ylabel("# of outlier perturbations")

In [ ]:
tf_analysis = n_outliers_per_tf.merge(tf_info, on="TF", how="left").fillna(0)
for _, row in tf_analysis.iterrows():
    print(
        f"TF {row['TF']}: {row['n_outliers']} outliers, "
        f"{row['n_edges']} edges, "
        f"{row['frac_uncertain']:.0%} uncertain, "
        f"{row['frac_nan']:.0%} no-confidence"
    )

In [ ]:
top = n_outliers_per_perturbation.sort_values("n_outliers", ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top["perturbation"], top["n_outliers"])
ax.set_xticklabels(top["perturbation"], rotation=45)
ax.set_ylabel("# outlier TFs")
ax.set_xlabel("Perturbation")

In [ ]:
perturbation_name = "rho"
df_tf_ = df_tf.loc[lambda x: x["perturbation"] == perturbation_name]
plt.hist(df_tf_["Zscore"], bins=100, density=True)

#### Emphasis on metabolic genes

In [ ]:
FBA_JSON = "/workspace/data/05142026_metabolome/iJO1366.json"
with open(FBA_JSON) as f:
    fba_data = json.load(f)
metabolic_genes = {g["name"].lower() for g in fba_data["genes"]}


df_tf_metabolic = df_tf[df_tf["perturbation"].isin(metabolic_genes)].copy()

In [ ]:
df_tf_metabolic_significant = df_tf_metabolic.loc[
    lambda x: x["abs_Zscore_emp"] >= 10
].copy()
df_tf_metabolic_significant.sort_values("abs_Zscore_emp", ascending=False).head(50)

In [ ]:
df_tf_metabolic_significant = df_tf_metabolic.loc[
    lambda x: x["abs_Zscore_emp"] >= 10
].copy()

deg_p = df_tf_metabolic_significant.groupby(
    "perturbation"
).size()  # = your promiscuity table
deg_t = df_tf_metabolic_significant.groupby("TF").size()  # = the mirror

df_tf_metabolic_significant["deg_p"] = df_tf_metabolic_significant.perturbation.map(
    deg_p
)
df_tf_metabolic_significant["deg_t"] = df_tf_metabolic_significant.TF.map(deg_t)

# low-low = best. Harmonic mean of degrees; small = specific on both sides.
df_tf_metabolic_significant["spec_score"] = (
    2
    * df_tf_metabolic_significant.deg_t
    * df_tf_metabolic_significant.deg_p
    / (df_tf_metabolic_significant.deg_t + df_tf_metabolic_significant.deg_p)
)
df_tf_metabolic_significant.sort_values("spec_score", ascending=True).head(100)

In [ ]:
df_tf_metabolic_significant.query("TF == 'dnaa'")